# Projet ML 

Ce notebook regroupe les différentes étapes de modélisation réalisées par l'équipe :
1. **Baseline validation** : Modèle de baseline de Régression Logistique sur un split temporel.
2. **Features** : Fonctions de feature engineering (calcul de statistiques sur le train et préparation des variables sans fuite temporelle).
3. **Modèle Final (LightGBM)** : Modèle final de classification LightGBM entraîné sur l'intégralité du train et appliqué sur le jeu de test pour générer les prédictions finales dans `submission.csv`.
4. **Prep** : Module de préparation commune des données pour faciliter d'autres expérimentations.

## 1. Baseline Validation

Dans cette section, nous chargeons les données d'entraînement, effectuons une analyse descriptive de base, réalisons un split temporel (80% train / 20% validation) et entraînons un modèle de baseline (Régression Logistique) avec un pipeline simple.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, classification_report, confusion_matrix

RANDOM_STATE = 42

# Chemin d'accès avec repli si exécuté depuis le sous-dossier Resultat
train_path = "reservations_train.csv"
if not os.path.exists(train_path):
    train_path = "../reservations_train.csv"

df = pd.read_csv(train_path)

TARGET_COL = "reservation_annulee"
DATE_COL = "date_reservation"
ID_COL = "reservation_id"

df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df["date_arrivee"] = pd.to_datetime(df["date_arrivee"])

print("Shape :", df.shape)
print("Taux d'annulation :", df[TARGET_COL].mean().round(4))

# Répartition cible
df[TARGET_COL].value_counts(normalize=True).plot(
    kind="bar", title="Répartition reservation_annulee")
plt.xticks([0, 1], ["Maintenue (0)", "Annulée (1)"], rotation=0)
plt.show()

print("\nTrain - date_reservation :", df[DATE_COL].min(), "->", df[DATE_COL].max())

# Valeurs manquantes
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("\nValeurs manquantes :\n", missing)
print("\nEn % :\n", (missing / len(df) * 100).round(1))

for c in ["hotel_id", "agent_id"]:
    print(c, "->", df[c].nunique(), "valeurs uniques")

# Boîtes à moustaches pour les variables numériques clés
num_check = ["delai_reservation_jours", "nuits", "montant_total_eur",
             "reservations_passees", "annulations_passees", "prix_moyen_nuit_eur"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.ravel(), num_check):
    df[c].plot(kind="box", ax=ax, title=c)
plt.tight_layout()
plt.show()

In [ ]:
def temporal_split(df, date_col, frac_train=0.8):
    """Trie par date et coupe en train_sub / val (PAS de shuffle, PAS de K-Fold)."""
    df_sorted = df.sort_values(date_col).reset_index(drop=True)
    idx = int(len(df_sorted) * frac_train)
    return df_sorted.iloc[:idx].copy(), df_sorted.iloc[idx:].copy()

train_sub, val = temporal_split(df, DATE_COL, frac_train=0.8)

print(f"\nTrain_sub : {train_sub.shape[0]} lignes "
      f"({train_sub[DATE_COL].min().date()} -> {train_sub[DATE_COL].max().date()})")
print(f"Val       : {val.shape[0]} lignes "
      f"({val[DATE_COL].min().date()} -> {val[DATE_COL].max().date()})")

y_train = train_sub[TARGET_COL]
y_val = val[TARGET_COL]

cols_to_drop = [TARGET_COL, DATE_COL, "date_arrivee", ID_COL, "hotel_id", "agent_id"]
X_train = train_sub.drop(columns=cols_to_drop)
X_val = val.drop(columns=cols_to_drop)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("\nColonnes numériques :", num_cols)
print("Colonnes catégorielles :", cat_cols)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),  
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols),
])

model_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])

model_lr.fit(X_train, y_train)

y_proba = model_lr.predict_proba(X_val)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

f1 = f1_score(y_val, y_pred, pos_label=1)
print(f"\n>>> F1-score BASELINE (seuil=0.5, classe annulation) : {f1:.4f}")
print("\nRapport de classification :")
print(classification_report(y_val, y_pred, digits=3))
print("Matrice de confusion :\n", confusion_matrix(y_val, y_pred))

## 2. Feature Engineering (Features)

Cette section contient les fonctions de feature engineering avancées :
- `fit_feature_stats` : calcule les statistiques d'entraînement (médianes, moyennes, quantiles de plafonnement) sur le sous-ensemble d'entraînement.
- `prepare_data` : applique ces transformations de manière robuste sans risque de fuite de données (anti-leakage) sur n'importe quel ensemble (train, validation, test).

In [ ]:
def fit_feature_stats(train_df: pd.DataFrame) -> dict:
    """
    Calcule toutes les statistiques nécessaires aux features, à partir du
    train UNIQUEMENT. Ce dictionnaire doit être réutilisé tel quel pour
    transformer la validation et le test.
    """
    stats = {}

    stats["prix_median_par_region_cat"] = (
        train_df.groupby(["region_hotel", "categorie_hotel"])["prix_moyen_nuit_eur"]
        .median()
    )
    stats["prix_median_global"] = train_df["prix_moyen_nuit_eur"].median()

    stats["prix_moyenne_par_region_cat"] = (
        train_df.groupby(["region_hotel", "categorie_hotel"])["prix_moyen_nuit_eur"]
        .mean()
    )
    stats["prix_moyenne_globale"] = train_df["prix_moyen_nuit_eur"].mean()

    stats["enfants_median"] = train_df["enfants"].median()
    stats["demandes_speciales_median"] = train_df["demandes_speciales"].median()

    ratio_tmp = train_df["nuits"] / train_df["chambres"].replace(0, np.nan)
    ratio_tmp = ratio_tmp.fillna(train_df["nuits"])
    stats["ratio_nuits_chambres_cap"] = ratio_tmp.quantile(0.99)

    def _ecart_prix_tmp(row):
        key = (row["region_hotel"], row["categorie_hotel"])
        moyenne = stats["prix_moyenne_par_region_cat"].get(key, stats["prix_moyenne_globale"])
        prix = row["prix_moyen_nuit_eur"]
        if pd.isna(prix):
            key2 = (row["region_hotel"], row["categorie_hotel"])
            prix = stats["prix_median_par_region_cat"].get(key2, stats["prix_median_global"])
        return prix - moyenne

    ecart_tmp = train_df.apply(_ecart_prix_tmp, axis=1)
    stats["ecart_prix_cap_low"] = ecart_tmp.quantile(0.01)
    stats["ecart_prix_cap_high"] = ecart_tmp.quantile(0.99)

    return stats

def prepare_data(df: pd.DataFrame, stats: dict) -> pd.DataFrame:
    """
    Applique le feature engineering à un dataframe donné, en utilisant
    uniquement les statistiques précalculées sur le train (`stats`).
    Ne modifie jamais `df` en place — retourne une copie.
    """
    out = df.copy()

    out["enfants"] = out["enfants"].fillna(stats["enfants_median"])
    out["demandes_speciales"] = out["demandes_speciales"].fillna(
        stats["demandes_speciales_median"]
    )

    def _impute_prix(row):
        if pd.notna(row["prix_moyen_nuit_eur"]):
            return row["prix_moyen_nuit_eur"]
        key = (row["region_hotel"], row["categorie_hotel"])
        return stats["prix_median_par_region_cat"].get(key, stats["prix_median_global"])

    out["prix_moyen_nuit_eur"] = out.apply(_impute_prix, axis=1)

    out["client_sans_historique"] = (out["reservations_passees"] == 0).astype(int)
    out["taux_annulation_historique"] = np.where(
        out["reservations_passees"] > 0,
        out["annulations_passees"] / out["reservations_passees"],
        0.0,
    )

    out["taille_groupe"] = out["adultes"] + out["enfants"]
    out["ratio_nuits_chambres"] = out["nuits"] / out["chambres"].replace(0, np.nan)
    out["ratio_nuits_chambres"] = out["ratio_nuits_chambres"].fillna(out["nuits"])
    out["ratio_nuits_chambres"] = out["ratio_nuits_chambres"].clip(
        upper=stats["ratio_nuits_chambres_cap"]
    )

    def _ecart_prix(row):
        key = (row["region_hotel"], row["categorie_hotel"])
        moyenne = stats["prix_moyenne_par_region_cat"].get(key, stats["prix_moyenne_globale"])
        return row["prix_moyen_nuit_eur"] - moyenne

    out["ecart_prix_vs_moyenne_region"] = out.apply(_ecart_prix, axis=1)
    out["ecart_prix_vs_moyenne_region"] = out["ecart_prix_vs_moyenne_region"].clip(
        lower=stats["ecart_prix_cap_low"], upper=stats["ecart_prix_cap_high"]
    )

    out["is_reservation_directe"] = out["agent_id"].isna().astype(int)
    out["tarif_remboursable_bin"] = (
        out["tarif_remboursable"].astype(str).str.lower().isin(["oui", "true", "1", "yes"])
    ).astype(int)
    out["acompte_x_remboursable"] = (
        out["type_acompte"].astype(str) + "_" + out["tarif_remboursable_bin"].astype(str)
    )

    out["a_modifie_reservation"] = (out["modifications_reservation"] > 0).astype(int)
    out["a_attendu_liste"] = (out["jours_liste_attente"] > 0).astype(int)

    out["mois_arrivee"] = out["date_arrivee"].dt.month
    out["jour_semaine_arrivee"] = out["date_arrivee"].dt.dayofweek

    bins = [-1, 7, 30, 90, np.inf]
    labels = ["derniere_minute", "court_terme", "moyen_terme", "long_terme"]
    out["delai_categorise"] = pd.cut(out["delai_reservation_jours"], bins=bins, labels=labels)

    return out

In [ ]:
# Test rapide des features définies
stats_demo = fit_feature_stats(train_sub)
train_sub_ready = prepare_data(train_sub, stats_demo)
print("Train prêt shape :", train_sub_ready.shape)
print(train_sub_ready[["taux_annulation_historique", "taille_groupe", "ecart_prix_vs_moyenne_region"]].head())

## 3. Modèle Final (LightGBM)

Cette section entraîne le modèle de production final (LightGBM Classifier) sur l'intégralité des données d'entraînement `reservations_train.csv` en utilisant les hyperparamètres optimaux (obtenus précédemment par optimisation Optuna) et le seuil de décision optimal de 0.51.
Les prédictions sur `reservations_test.csv` sont ensuite générées et sauvegardées sous le nom `submission.csv`.

In [ ]:
import lightgbm as lgb

# Chemins d'accès avec repli si exécuté depuis le sous-dossier Resultat
test_path = "reservations_test.csv"
if not os.path.exists(test_path):
    test_path = "../reservations_test.csv"

train_df = pd.read_csv(train_path, parse_dates=[DATE_COL, "date_arrivee"])
test_df = pd.read_csv(test_path, parse_dates=[DATE_COL, "date_arrivee"])

# Ajustement des stats sur TOUTES les données d'entraînement
stats = fit_feature_stats(train_df)
train_ready = prepare_data(train_df, stats)
test_ready = prepare_data(test_df, stats)

y_train = train_ready[TARGET_COL]
COLS_TO_DROP_BASE = [TARGET_COL, DATE_COL, "date_arrivee", ID_COL, "hotel_id", "agent_id"]
X_train = train_ready.drop(columns=COLS_TO_DROP_BASE)
X_test = test_ready.drop(columns=[c for c in COLS_TO_DROP_BASE if c != TARGET_COL])

# Encodage catégoriel pour LightGBM
cat_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
for c in cat_cols:
    X_train[c] = X_train[c].astype("category")
    cats = X_train[c].cat.categories
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=cats)

# Hyperparamètres optimaux et seuil optimal
BEST_PARAMS = {
    'n_estimators': 256, 'learning_rate': 0.014464020877451817, 'num_leaves': 36,
    'max_depth': 4, 'min_child_samples': 59, 'subsample': 0.7004229904782565,
    'colsample_bytree': 0.7724567389301411, 'reg_alpha': 0.0016742237510088004,
    'reg_lambda': 0.13601246953903182, 'scale_pos_weight': 3.6629470290120234,
}
BEST_THRESHOLD = 0.51

params = dict(BEST_PARAMS)
params.update({"objective": "binary", "metric": "None", "verbosity": -1,
                "boosting_type": "gbdt", "random_state": RANDOM_STATE})

model_lgb = lgb.LGBMClassifier(**params)
model_lgb.fit(X_train, y_train, categorical_feature=cat_cols)

proba_test = model_lgb.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= BEST_THRESHOLD).astype(int)

result = pd.DataFrame({
    "reservation_id": test_ready[ID_COL].values,
    "probabilite_annulation": proba_test,
    "reservation_annulee": pred_test,
})

# Sauvegarde du fichier sous le nom submission.csv
# Si exécuté depuis le dossier racine du projet, on enregistre dans le dossier Resultat
if os.path.exists("Resultat"):
    submission_path = "Resultat/submission.csv"
else:
    submission_path = "submission.csv"

result.to_csv(submission_path, index=False)
print("\nModèle final : LightGBM")
print("Prédictions générées et enregistrées sous :", submission_path)
print("Taux d'annulation prédit sur le test :", round(result['reservation_annulee'].mean(), 4))
print(result.head(10))

## 4. Préparation Commune (Prep)

Cette dernière section implémente le module de préparation commune des données (`prep_common.py`) qui permet de charger les données et d'appliquer le split temporel ainsi que le feature engineering de façon modulaire.

In [ ]:
def load_and_prepare():
    train_path = "reservations_train.csv"
    test_path = "reservations_test.csv"
    if not os.path.exists(train_path):
        train_path = "../reservations_train.csv"
    if not os.path.exists(test_path):
        test_path = "../reservations_test.csv"
        
    train_df = pd.read_csv(train_path, parse_dates=[DATE_COL, "date_arrivee"])
    test_df = pd.read_csv(test_path, parse_dates=[DATE_COL, "date_arrivee"])

    # 1. Split temporel AVANT tout calcul de stats (anti-fuite)
    train_sub_raw, val_raw = temporal_split(train_df, DATE_COL, frac_train=0.8)

    # 2. Stats de feature engineering calculées UNIQUEMENT sur train_sub
    stats = fit_feature_stats(train_sub_raw)

    # 3. Application des mêmes stats partout
    train_sub = prepare_data(train_sub_raw, stats)
    val = prepare_data(val_raw, stats)
    test_ready = prepare_data(test_df, stats)

    y_train = train_sub[TARGET_COL]
    y_val = val[TARGET_COL]

    X_train = train_sub.drop(columns=COLS_TO_DROP_BASE)
    X_val = val.drop(columns=COLS_TO_DROP_BASE)
    X_test = test_ready.drop(columns=[c for c in COLS_TO_DROP_BASE if c != TARGET_COL])

    return X_train, y_train, X_val, y_val, X_test, test_ready[ID_COL]

X_train_prep, y_train_prep, X_val_prep, y_val_prep, X_test_prep, test_ids_prep = load_and_prepare()
print("\nVérification de la préparation commune (load_and_prepare) :")
print("X_train_prep :", X_train_prep.shape)
print("X_val_prep   :", X_val_prep.shape)
print("X_test_prep  :", X_test_prep.shape)